In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch
import torchvision
import quantus
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse


import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from scipy.stats import spearmanr
from tqdm import tqdm

import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:

highest_corrs_absolute_power = np.load('highest_corrs_absolute_power.npy', allow_pickle=True).item()

In [ ]:
highest_corss_absolute_power_abs = np.load('highest_corrs_absolute_power_abs.npy', allow_pickle=True).item()
highest_corss_relative_power_abs = np.load('highest_corrs_relative_power_abs.npy', allow_pickle=True).item()


highest_corrs_periodic_abs = np.load('all_subjects_highest_corrs_abs_perodic.npy', allow_pickle=True).item()

highest_corrs_absolute_power = np.load('highest_corrs_absolute_power.npy', allow_pickle=True).item()
highest_corrs_relative_power = np.load('highest_corrs_relative_power.npy', allow_pickle=True).item()
#highest_corrs_fractal_exponent = np.load('highest_corrs_fractal_exponent.npy', allow_pickle=True).item()

highest_corrs_periodic = np.load('all_subjects_highests_corrs_periodic.npy', allow_pickle=True).item()

#highest_corrs_channel_wise_connectivity = np.load("highest_corss_channel_wise_functional_connectivity.npy", allow_pickle=True).item()
#top_k_per_subject = np.load("top_k_dict.npy", allow_pickle=True).item()


In [ ]:
highest_corrs_periodic

In [ ]:
top_k_per_subject_abs = np.load("all_subject_channel_importances_gradshap_abs.npy", allow_pickle=True).item()
top_k_per_subject = np.load("all_subject_channel_importances_gradshap.npy", allow_pickle=True).item()

In [ ]:
highest_corrs_fractal_offset = np.load('highest_corrs_fractal_offset.npy', allow_pickle=True).item()
highest_corrs_fractal_offset_abs = np.load('highest_corrs_abs_fractal_offset.npy', allow_pickle=True).item()
highest_corrs_fractal_exponent_abs = np.load('highest_corrs_abs_fractal_exponent.npy', allow_pickle=True).item()
highest_corrs_fractal_exponent = np.load('highest_corrs_fractal_exponent.npy', allow_pickle=True).item()

In [ ]:
def extract_top_10_channels_and_mean_corr(all_subjects_corrs, k=60):
    top_10_channels_per_subject = {}
    mean_corr_per_subject = {}
    
    for subject, channels in all_subjects_corrs.items():
        sorted_channels = sorted(channels.items(), key=lambda item: abs(item[1]), reverse=True)[:k]
        top_10_channels_per_subject[subject] = dict(sorted_channels)
        
        mean_corr = np.mean([abs(corr) for ch, corr in sorted_channels])
        mean_corr_per_subject[subject] = mean_corr
    
    return top_10_channels_per_subject, mean_corr_per_subject

top_10_per_subject_dict_abs = {subject_index: top_10_channels for subject_index, top_10_channels in zip(cfg.dataset.test_subject_indices, top_10_per_subject_abs)}

top_10_per_subject_dict = {subject_index: top_10_channels for subject_index, top_10_channels in zip(cfg.dataset.test_subject_indices, top_10_per_subject)}

top_10_channels_abs_power, mean_corr_abs_power = extract_top_10_channels_and_mean_corr_freqband(highest_corss_absolute_power)
top_10_channels_relative_power, mean_corr_relative_power = extract_top_10_channels_and_mean_corr_freqband(highest_corss_relative_power)
top_10_channels_periodic, mean_corr_periodic = extract_top_10_channels_and_mean_corr_freqband(highest_corrs_periodic)
top_10_channels_fractal, mean_corr_fractal = extract_top_10_channels_and_mean_corr(highest_corrs_fractal)
top_10_channels_periodic, mean_corr_periodic = extract_top_10_channels_and_mean_corr_freqband(highest_corrs_periodic)
#top_10_channels_explanation_func = extract_top_10_channels_and_mean_corr(highest_corrs_channel_wise_connectivity)

print("Top 10 channels for absolute power:", top_10_channels_abs_power)
print("Mean correlation for absolute power:", mean_corr_abs_power)
print("Top 10 channels for relative power:", top_10_channels_relative_power)
print("Mean correlation for relative power:", mean_corr_relative_power)
print("Top 10 channels for periodic:", top_10_channels_periodic)
print("Mean correlation for periodic:", mean_corr_periodic)
print("Top 10 channels for fractal:", top_10_channels_fractal)
print("Mean correlation for fractal:", mean_corr_fractal)
#print("Top 10 channels for explanation function:", top_10_channels_explanation_func)


# plot significant and top channels next to each other

In [ ]:
def compare_top_k_and_significant(top_k_abs, highest_corss_absolute_power, highest_corss_relative_power, highest_corrs_fractal, highest_corrs_periodic,  frequency_band='gamma'):

    cfg = load_config()
    all_fig, all_axs = plt.subplots(nrows=len(cfg.dataset.test_subject_indices), ncols=5, figsize=(14,2*len(cfg.dataset.test_subject_indices)))
    all_fig.suptitle(f"Top K and significant channels for each subject in {frequency_band} band")
    all_importances = np.zeros((len(cfg.dataset.test_subject_indices), 5, 60))
    for si, subject_index in enumerate(cfg.dataset.test_subject_indices):

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info

        print(f"Subject {subject_index}:")
        #print("Top k channels:", top_k_per_subject[subject_index])
        print("top k abs channels:", top_k_per_subject_abs[subject_index])
        print("highest_corrs_absolute_power:", highest_corss_absolute_power[subject_index][frequency_band])
        print("highest_corss_relative_power:", highest_corss_relative_power[subject_index][frequency_band])
        print("highest_corrs_fractal:", highest_corrs_fractal[subject_index])
        print("highest_corrs_periodic:", highest_corrs_periodic[subject_index][frequency_band])
        #print("highest_corrs_channel_wise_connectivity:", highest_corrs_channel_wise_connectivity[subject_index][frequency_band])

        #flattened_array = [item for sublist in list(map(lambda x: x.split("_"), highest_corrs_channel_wise_connectivity[subject_index][frequency_band].keys())) for item in sublist]
        #print("flattened_array:", flattened_array)
        cfg = load_config()
        cfg.dataset.subject_index = subject_index
   
        _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)

        sig_ch_data_absolute_power = np.zeros(len(ch_names))
        sig_ch_data_relative_power = np.zeros(len(ch_names))
        sig_ch_data_fractal = np.zeros(len(ch_names))
        sig_ch_data_periodic = np.zeros(len(ch_names))
        #sig_ch_data_connectivity = np.zeros(len(ch_names))
        #top_k_ch = np.zeros(len(ch_names))
        top_k_abs_ch = np.zeros(len(ch_names))

        for idx, ch in enumerate(ch_names):
            if ch in highest_corss_absolute_power[subject_index][frequency_band].keys():
                sig_ch_data_absolute_power[idx] = 1
            if ch in highest_corss_relative_power[subject_index][frequency_band].keys():
                sig_ch_data_relative_power[idx] = 1
            if ch in highest_corrs_fractal[subject_index].keys():
                sig_ch_data_fractal[idx] = 1
            if ch in highest_corrs_periodic[subject_index][frequency_band].keys():
                sig_ch_data_periodic[idx] = 1
            if ch in top_k_abs[subject_index]:
                top_k_abs_ch[idx] = 1
        
        all_importances[si] = np.array([sig_ch_data_absolute_power, sig_ch_data_relative_power, sig_ch_data_fractal, sig_ch_data_periodic, top_k_abs_ch])

        mne.viz.plot_topomap(sig_ch_data_absolute_power, info_subj , show = False, contours=4, ch_type='eeg', axes=all_axs[si,0])
        all_axs[si,0].set_title("Absolute Power")
        mne.viz.plot_topomap(sig_ch_data_relative_power, info_subj , show = False, contours=4, ch_type='eeg', axes=all_axs[si,1])
        all_axs[si,1].set_title("Relative Power")
        mne.viz.plot_topomap(sig_ch_data_fractal, info_subj , show = False, contours=4, ch_type='eeg', axes=all_axs[si,2])
        all_axs[si,2].set_title("Fractal exponent")
        mne.viz.plot_topomap(sig_ch_data_periodic, info_subj , show = False, contours=4, ch_type='eeg', axes=all_axs[si,3])
        all_axs[si,3].set_title("Periodic component")
        mne.viz.plot_topomap(top_k_abs_ch, info_subj , show = False, contours=4, ch_type='eeg', axes=all_axs[si,4])
        all_axs[si,4].set_title("Top K abs")
        all_fig.savefig(f"top_k_and_significant_channels_{frequency_band}.png")
        
    fig,axs = plt.subplots(nrows=1, ncols=5, figsize=(14,2))
    fig.suptitle(f"Average of all subjects in {frequency_band} band")
    all_importances = np.sum(all_importances, axis=0)
    for i in range(5):
         mne.viz.plot_topomap(all_importances[i], info_subj , show = False, contours=4, ch_type='eeg', axes=axs[i])
    fig.subplots_adjust(wspace=0.1)
    fig.savefig(f"top_k_and_significant_channels_{frequency_band}_average.png")
    return all_importances, info_subj



In [ ]:
def compare_top_k_and_significant2(top_k_abs, top_k, highest_corss_absolute_power, highest_corss_relative_power, highest_corrs_fractal, highest_corrs_periodic, frequency_band='gamma', subfig=None):

    cfg = load_config()
    all_fig, all_axs = plt.subplots(nrows=len(cfg.dataset.test_subject_indices), ncols=6, figsize=(16, 2*len(cfg.dataset.test_subject_indices)))
    all_fig.suptitle(f"Top K and significant channels for each subject in {frequency_band} band")
    all_importances = np.zeros((len(cfg.dataset.test_subject_indices), 6, 60))
    for si, subject_index in enumerate(cfg.dataset.test_subject_indices):

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info

        print(f"Subject {subject_index}:")
        print("top k abs channels:", top_k_abs[subject_index])
        print("top k channels:", top_k[subject_index])
        print("highest_corrs_absolute_power:", highest_corss_absolute_power[subject_index][frequency_band])
        print("highest_corss_relative_power:", highest_corss_relative_power[subject_index][frequency_band])
        print("highest_corrs_fractal:", highest_corrs_fractal[subject_index])
        print("highest_corrs_periodic:", highest_corrs_periodic[subject_index][frequency_band])

        cfg = load_config()
        cfg.dataset.subject_index = subject_index
   
        _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)

        sig_ch_data_absolute_power = np.zeros(len(ch_names))
        sig_ch_data_relative_power = np.zeros(len(ch_names))
        sig_ch_data_fractal = np.zeros(len(ch_names))
        sig_ch_data_periodic = np.zeros(len(ch_names))
        top_k_abs_ch = np.zeros(len(ch_names))
        top_k_ch = np.zeros(len(ch_names))

        for idx, ch in enumerate(ch_names):
            if ch in highest_corss_absolute_power[subject_index][frequency_band].keys():
                sig_ch_data_absolute_power[idx] = 1
            if ch in highest_corss_relative_power[subject_index][frequency_band].keys():
                sig_ch_data_relative_power[idx] = 1
            if ch in highest_corrs_fractal[subject_index].keys():
                sig_ch_data_fractal[idx] = 1
            if ch in highest_corrs_periodic[subject_index][frequency_band].keys():
                sig_ch_data_periodic[idx] = 1
            if ch in top_k_abs[subject_index]:
                top_k_abs_ch[idx] = 1
            if ch in top_k[subject_index]:
                top_k_ch[idx] = 1
        
        all_importances[si] = np.array([sig_ch_data_absolute_power, sig_ch_data_relative_power, sig_ch_data_fractal, sig_ch_data_periodic, top_k_abs_ch, top_k_ch])

        mne.viz.plot_topomap(sig_ch_data_absolute_power, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 0])
        all_axs[si, 0].set_title("Absolute Power")
        mne.viz.plot_topomap(sig_ch_data_relative_power, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 1])
        all_axs[si, 1].set_title("Relative Power")
        mne.viz.plot_topomap(sig_ch_data_fractal, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 2])
        all_axs[si, 2].set_title("Fractal exponent")
        mne.viz.plot_topomap(sig_ch_data_periodic, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 3])
        all_axs[si, 3].set_title("Periodic component")
        mne.viz.plot_topomap(top_k_abs_ch, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 4])
        all_axs[si, 4].set_title("Top K abs")
        mne.viz.plot_topomap(top_k_ch, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 5])
        all_axs[si, 5].set_title("Top K")

    all_fig.savefig(f"top_k_and_significant_channels_{frequency_band}.png")
    all_importances = np.sum(all_importances, axis=0)
    if subfig is None:
        fig, axs = plt.subplots(nrows=1, ncols=6, figsize=(16, 2))
        fig.suptitle(f"Average of all subjects in {frequency_band} band", fontsize=17)
        
        for i in range(6):
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
        fig.subplots_adjust(wspace=0.1)
        fig.savefig(f"top_k_and_significant_channels_{frequency_band}_average.png")
    else:
        
        axs = subfig.subplots(nrows=1, ncols=6, figsize=(12, 4))
        for i in range(6):
             if frequency_band == 'theta':
                axs[0].set_title("Absolute power", fontsize=12)
                axs[1].set_title("Relative power", fontsize=12)
                axs[2].set_title("Fractal exponent", fontsize=12)
                axs[3].set_title("Periodic component", fontsize=12)
                axs[4].set_title("Top K abs", fontsize=12)
                axs[5].set_title("Top K", fontsize=12)
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
    return all_importances, info_subj



In [ ]:
def compare_power_features(highest_corss_absolute_power, highest_corss_relative_power, highest_corrs_periodic, frequency_band='gamma', subfig=None):

    cfg = load_config()
    all_fig, all_axs = plt.subplots(nrows=len(cfg.dataset.test_subject_indices), ncols=3, figsize=(12, 2*len(cfg.dataset.test_subject_indices)))
    all_fig.suptitle(f"Top K and significant channels for each subject in {frequency_band} band")
    all_importances = np.zeros((len(cfg.dataset.test_subject_indices), 3, 60))
    for si, subject_index in enumerate(cfg.dataset.test_subject_indices):

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info

        print(f"Subject {subject_index}:")
        print("highest_corrs_absolute_power:", highest_corss_absolute_power[subject_index][frequency_band])
        print("highest_corss_relative_power:", highest_corss_relative_power[subject_index][frequency_band])
        print("highest_corrs_periodic:", highest_corrs_periodic[subject_index][frequency_band])

        cfg = load_config()
        cfg.dataset.subject_index = subject_index
   
        _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)

        sig_ch_data_absolute_power = np.zeros(len(ch_names))
        sig_ch_data_relative_power = np.zeros(len(ch_names))
        sig_ch_data_periodic = np.zeros(len(ch_names))

        for idx, ch in enumerate(ch_names):
            if ch in highest_corss_absolute_power[subject_index][frequency_band].keys():
                sig_ch_data_absolute_power[idx] = 1
            if ch in highest_corss_relative_power[subject_index][frequency_band].keys():
                sig_ch_data_relative_power[idx] = 1
            if ch in highest_corrs_periodic[subject_index][frequency_band].keys():
                sig_ch_data_periodic[idx] = 1
        
        all_importances[si] = np.array([sig_ch_data_absolute_power, sig_ch_data_relative_power, sig_ch_data_periodic])

        mne.viz.plot_topomap(sig_ch_data_absolute_power, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 0])
        all_axs[si, 0].set_title("Absolute Power")
        mne.viz.plot_topomap(sig_ch_data_relative_power, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 1])
        all_axs[si, 1].set_title("Relative Power")
        mne.viz.plot_topomap(sig_ch_data_periodic, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 2])
        all_axs[si, 2].set_title("Periodic component")

    all_fig.savefig(f"top_k_and_significant_channels_{frequency_band}.png")
    all_importances = np.sum(all_importances, axis=0)
    if subfig is None:
        

        fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12, 2))
        fig.subplots_adjust(wspace=-0.5)
        fig.suptitle(f"Average of all subjects in {frequency_band} band", y=1.1, fontsize=18)
        if frequency_band == 'theta':
            axs[0].set_title("Absolute power", fontsize=15, y=0.95)
            axs[1].set_title("Relative power", fontsize=15, y=0.95)
            axs[2].set_title("Periodic component", fontsize=15, y=0.95)
        
        for i in range(3):
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
        fig.savefig(f"top_k_and_significant_channels_{frequency_band}_average.png")
    else:
        
        axs = subfig.subplots(nrows=1, ncols=3)
        for i in range(3):
             if frequency_band == 'theta':
                axs[0].set_title("Absolute power", fontsize=16)
                axs[1].set_title("Relative power", fontsize=16)
                axs[2].set_title("Periodic component", fontsize=16)
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
    return all_importances, info_subj



In [ ]:
def compare_power_features_all_channels(highest_corss_absolute_power, highest_corss_relative_power, highest_corrs_periodic, frequency_band='gamma', subfig=None):

    cfg = load_config()
    all_fig, all_axs = plt.subplots(nrows=len(cfg.dataset.test_subject_indices), ncols=3, figsize=(12, 2*len(cfg.dataset.test_subject_indices)))
    all_fig.suptitle(f"Top K and significant channels for each subject in {frequency_band} band")
    all_importances = np.zeros((len(cfg.dataset.test_subject_indices), 3, 60))
    for si, subject_index in enumerate(cfg.dataset.test_subject_indices):

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info

        print(f"Subject {subject_index}:")
        print("highest_corrs_absolute_power:", highest_corss_absolute_power[subject_index][frequency_band])
        print("highest_corss_relative_power:", highest_corss_relative_power[subject_index][frequency_band])
        print("highest_corrs_periodic:", highest_corrs_periodic[subject_index][frequency_band])

        cfg = load_config()
        n_subjects = len(cfg.dataset.test_subject_indices)
        cfg.dataset.subject_index = subject_index
   
        _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)



        sum_sig_ch_data_absolute_power = 0
        sum_sig_ch_data_relative_power = 0
        sum_sig_ch_data_periodic = 0

        for idx, ch in enumerate(ch_names):
            if ch in highest_corss_absolute_power[subject_index][frequency_band].keys():
                sum_sig_ch_data_absolute_power += highest_corss_absolute_power[subject_index][frequency_band][ch]
            if ch in highest_corss_relative_power[subject_index][frequency_band].keys():
                sum_sig_ch_data_relative_power+= highest_corss_relative_power[subject_index][frequency_band][ch]
            if ch in highest_corrs_periodic[subject_index][frequency_band].keys():
                sum_sig_ch_data_periodic+= highest_corrs_periodic[subject_index][frequency_band][ch]
        

        sig_ch_data_absolute_power = np.zeros(len(ch_names))
        sig_ch_data_relative_power = np.zeros(len(ch_names))
        sig_ch_data_periodic = np.zeros(len(ch_names))
        for idx, ch in enumerate(ch_names):

            if ch in highest_corss_absolute_power[subject_index][frequency_band].keys():
                sig_ch_data_absolute_power[idx] += (highest_corss_absolute_power[subject_index][frequency_band][ch] /sum_sig_ch_data_absolute_power)
            if ch in highest_corss_relative_power[subject_index][frequency_band].keys():
                sig_ch_data_relative_power[idx] += (highest_corss_relative_power[subject_index][frequency_band][ch] / sum_sig_ch_data_relative_power)
            if ch in highest_corrs_periodic[subject_index][frequency_band].keys():
                sig_ch_data_periodic[idx] += (highest_corrs_periodic[subject_index][frequency_band][ch] / sum_sig_ch_data_periodic)
        
        all_importances[si] = np.array([sig_ch_data_absolute_power, sig_ch_data_relative_power, sig_ch_data_periodic])

        mne.viz.plot_topomap(sig_ch_data_absolute_power, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 0])
        all_axs[si, 0].set_title("Absolute Power")
        mne.viz.plot_topomap(sig_ch_data_relative_power, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 1])
        all_axs[si, 1].set_title("Relative Power")
        mne.viz.plot_topomap(sig_ch_data_periodic, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 2])
        all_axs[si, 2].set_title("Periodic component")


    all_fig.savefig(f"top_k_and_significant_channels_{frequency_band}.png")
    all_importances = np.sum(all_importances, axis=0)/n_subjects
    if subfig is None:
        

        fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12, 2))
        fig.subplots_adjust(wspace=-0.5)
        fig.suptitle(f"Average of all subjects in {frequency_band} band", y=1.1, fontsize=18)
        if frequency_band == 'theta':
            axs[0].set_title("Absolute power", fontsize=15, y=0.95)
            axs[1].set_title("Relative power", fontsize=15, y=0.95)
            axs[2].set_title("Periodic component", fontsize=15, y=0.95)
        
        for i in range(3):
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
        fig.savefig(f"top_k_and_significant_channels_{frequency_band}_average_all_channels.png")
    else:
        
        axs = subfig.subplots(nrows=1, ncols=3)
        for i in range(3):
             if frequency_band == 'theta':
                axs[0].set_title("Absolute power", fontsize=16)
                axs[1].set_title("Relative power", fontsize=16)
                axs[2].set_title("Periodic component", fontsize=16)
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
    return all_importances, info_subj



In [ ]:
def compare_power_features_all_channels(highest_corss_absolute_power, highest_corss_relative_power, highest_corrs_periodic, frequency_band='gamma', subfig=None, sig_level=0.05, normalize=False):

    cfg = load_config()
    all_fig, all_axs = plt.subplots(nrows=len(cfg.dataset.test_subject_indices), ncols=3, figsize=(12, 2*len(cfg.dataset.test_subject_indices)))
    all_fig.suptitle(f"Top K and significant channels for each subject in {frequency_band} band")
    all_importances = np.zeros((len(cfg.dataset.test_subject_indices), 3, 60))
    for si, subject_index in enumerate(cfg.dataset.test_subject_indices):

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info

        print(f"Subject {subject_index}:")
        print("highest_corrs_absolute_power:", highest_corss_absolute_power[subject_index][frequency_band])
        print("highest_corss_relative_power:", highest_corss_relative_power[subject_index][frequency_band])
        print("highest_corrs_periodic:", highest_corrs_periodic[subject_index][frequency_band])

        cfg = load_config()
        n_subjects = len(cfg.dataset.test_subject_indices)
        cfg.dataset.subject_index = subject_index
   
        _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)



        sum_sig_ch_data_absolute_power = 0
        sum_sig_ch_data_relative_power = 0
        sum_sig_ch_data_periodic = 0

        if normalize:
            for idx, ch in enumerate(ch_names):
                if ch in highest_corss_absolute_power[subject_index][frequency_band].keys():
                    sum_sig_ch_data_absolute_power += highest_corss_absolute_power[subject_index][frequency_band][ch]
                if ch in highest_corss_relative_power[subject_index][frequency_band].keys():
                    sum_sig_ch_data_relative_power+= highest_corss_relative_power[subject_index][frequency_band][ch]
                if ch in highest_corrs_periodic[subject_index][frequency_band].keys():
                    sum_sig_ch_data_periodic+= highest_corrs_periodic[subject_index][frequency_band][ch]
        

        sig_ch_data_absolute_power = np.zeros(len(ch_names))
        sig_ch_data_relative_power = np.zeros(len(ch_names))
        sig_ch_data_periodic = np.zeros(len(ch_names))
        for idx, ch in enumerate(ch_names):

            if ch in highest_corss_absolute_power[subject_index][frequency_band].keys():
                if highest_corss_absolute_power[subject_index][frequency_band][ch]["pval"]<sig_level:
                    sig_ch_data_absolute_power[idx] += highest_corss_absolute_power[subject_index][frequency_band][ch]["stat"] #/sum_sig_ch_data_absolute_power)
            if ch in highest_corss_relative_power[subject_index][frequency_band].keys():
                if highest_corss_relative_power[subject_index][frequency_band][ch]["pval"]<sig_level:
                    sig_ch_data_relative_power[idx] += highest_corss_relative_power[subject_index][frequency_band][ch]["stat"]  #/ sum_sig_ch_data_relative_power)
                
            if ch in highest_corrs_periodic[subject_index][frequency_band].keys():
                if highest_corrs_periodic[subject_index][frequency_band][ch]["pval"]<sig_level:
                    sig_ch_data_periodic[idx] += highest_corrs_periodic[subject_index][frequency_band][ch]["stat"] #/ sum_sig_ch_data_periodic)
        
        all_importances[si] = np.array([sig_ch_data_absolute_power, sig_ch_data_relative_power, sig_ch_data_periodic])

        mne.viz.plot_topomap(sig_ch_data_absolute_power, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 0])
        all_axs[si, 0].set_title("Absolute Power")
        mne.viz.plot_topomap(sig_ch_data_relative_power, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 1])
        all_axs[si, 1].set_title("Relative Power")
        mne.viz.plot_topomap(sig_ch_data_periodic, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 2])
        all_axs[si, 2].set_title("Periodic component")


    all_fig.savefig(f"top_k_and_significant_channels_{frequency_band}.png")
    all_importances = np.sum(all_importances, axis=0)/n_subjects
    if subfig is None:
        

        fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12, 2))
        fig.subplots_adjust(wspace=-0.5)
        fig.suptitle(f"Average of all subjects in {frequency_band} band", y=1.1, fontsize=18)
        if frequency_band == 'delta':
            axs[0].set_title("Absolute power", fontsize=16, y=1.08)
            axs[1].set_title("Relative power", fontsize=16, y=1.08)
            axs[2].set_title("Periodic component", fontsize=16, y=1.08)
        
        for i in range(3):
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
        fig.savefig(f"top_k_and_significant_channels_{frequency_band}_average_all_channels.png")
    else:
        
        axs = subfig.subplots(nrows=1, ncols=3)
        for i in range(3):
             if frequency_band == 'delta':
                axs[0].set_title("Absolute power", fontsize=18, y=1.1)
                axs[1].set_title("Relative power", fontsize=18, y=1.1)
                axs[2].set_title("Periodic component power", fontsize=18, y=1.1)
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
    return all_importances, info_subj



In [ ]:
def compare_top_k_and_fractal(top_k_abs, highest_corrs_fractal_offset, highest_corrs_fractal_exponent, subfig=None, sig_level=0.05):

    cfg = load_config()
    #all_fig, all_axs = plt.subplots(nrows=len(cfg.dataset.test_subject_indices), ncols=3, figsize=(12, 2*len(cfg.dataset.test_subject_indices)))
    #all_fig.suptitle(f"Top K and significant channels for each subject in {frequency_band} band")
    all_importances = np.zeros((len(cfg.dataset.test_subject_indices), 3, 60))
    for si, subject_index in enumerate(cfg.dataset.test_subject_indices):

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info

        ##print(f"Subject {subject_index}:")
        #print("top k abs channels:", top_k_abs[subject_index])
        #print("top k channels:", top_k[subject_index])
        #print("highest_corrs_fractal:", highest_corrs_fractal[subject_index])

        cfg = load_config()
        cfg.dataset.subject_index = subject_index
   
        _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)

        sig_ch_data_fractal_exponent = np.zeros(len(ch_names))
        sig_ch_data_fractal_offset = np.zeros(len(ch_names))
        top_k_abs_ch = np.zeros(len(ch_names))

        for idx, ch in enumerate(ch_names):
            if ch in highest_corrs_fractal_exponent[subject_index].keys():
                if highest_corrs_fractal_exponent[subject_index][ch]["pval"]<sig_level:
                    sig_ch_data_fractal_exponent[idx] += highest_corrs_fractal_exponent[subject_index][ch]["stat"]
            if ch in highest_corrs_fractal_offset[subject_index].keys():
                if highest_corrs_fractal_offset[subject_index][ch]["pval"]<sig_level:
                    sig_ch_data_fractal_offset[idx] += highest_corrs_fractal_offset[subject_index][ch]["stat"]
            if ch in top_k_abs[subject_index]:
                top_k_abs_ch[idx] += top_k_abs[subject_index][ch]
                
        all_importances[si] = np.array([sig_ch_data_fractal_exponent, sig_ch_data_fractal_offset, top_k_abs_ch])

        #mne.viz.plot_topomap(sig_ch_data_fractal_exponent, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 0])
        #all_axs[si, 0].set_title("Fractal exponent")
        #mne.viz.plot_topomap(sig_ch_data_fractal_offset, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 1])
        #all_axs[si, 1].set_title("Fractal offset")
        #mne.viz.plot_topomap(top_k_abs_ch, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 2])
        #
        # all_axs[si, 2].set_title("Top K absolute")

    #all_fig.savefig(f"top_k_and_significant_channels_{frequency_band}.png")
    all_importances = np.average(all_importances, axis=0)
    if subfig is None:
        fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12, 3))
        fig.subplots_adjust(wspace=0.01)
        #fig.suptitle(f"Average of all subjects in {frequency_band} band")
        names = ["Fractal exponent", "Fractal offset", "Top K abs"]
        for i in range(3):
            mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
            axs[i].set_title(names[i], fontsize=15)
        #fig.savefig(f"top_k_and_significant_channels_{frequency_band}_average.png")

        fig.savefig("all_topk_average.png")
        
    else:
        
        for i in range(3):
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
    return fig



def compare_top_k_and_fractal_all_channels(top_k_abs, highest_corrs_fractal_offset, highest_corrs_fractal_exponent, subfig=None):

    cfg = load_config()
    all_fig, all_axs = plt.subplots(nrows=len(cfg.dataset.test_subject_indices), ncols=3, figsize=(12, 2*len(cfg.dataset.test_subject_indices)))
    #all_fig.suptitle(f"Top K and significant channels for each subject in {frequency_band} band")
    all_importances = np.zeros((len(cfg.dataset.test_subject_indices), 3, 60))
    for si, subject_index in enumerate(cfg.dataset.test_subject_indices):

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info

        #print(f"Subject {subject_index}:")
        #print("top k abs channels:", top_k_abs[subject_index])
        #print("top k channels:", top_k[subject_index])
        #print("highest_corrs_fractal:", highest_corrs_fractal[subject_index])

        cfg = load_config()
        cfg.dataset.subject_index = subject_index
   
        _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)

        sig_ch_data_fractal = np.zeros(len(ch_names))
        top_k_abs_ch = np.zeros(len(ch_names))
        top_k_ch = np.zeros(len(ch_names))

        for idx, ch in enumerate(ch_names):
            if ch in highest_corrs_fractal[subject_index].keys():
                
                sig_ch_data_fractal[idx] = 1
            if ch in top_k_abs[subject_index]:
                top_k_abs_ch[idx] = 1
            if ch in top_k[subject_index]:
                top_k_ch[idx] = 1
        
        all_importances[si] = np.array([sig_ch_data_fractal, top_k_abs_ch, top_k_ch])

        mne.viz.plot_topomap(sig_ch_data_fractal, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 0])
        all_axs[si, 0].set_title("Fractal exponent")
        mne.viz.plot_topomap(top_k_abs_ch, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 1])
        all_axs[si, 1].set_title("Top K abs")
        mne.viz.plot_topomap(top_k_ch, info_subj, show=False, contours=4, ch_type='eeg', axes=all_axs[si, 2])
        all_axs[si, 2].set_title("Top K")

    #all_fig.savefig(f"top_k_and_significant_channels_{frequency_band}.png")
    all_importances = np.average(all_importances, axis=0)
    if subfig is None:
        fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(12, 2))
        fig.subplots_adjust(wspace=-0.5)
        #fig.suptitle(f"Average of all subjects in {frequency_band} band")
        names = ["Fractal exponent", "Top K abs", "Top K"]
        for i in range(3):
            mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
            axs[i].set_title(names[i], fontsize=15)
        #fig.savefig(f"top_k_and_significant_channels_{frequency_band}_average.png")

        fig.savefig("all_topk_average.png")
        
    else:
        
        
        for i in range(3):
             mne.viz.plot_topomap(all_importances[i], info_subj, show=False, contours=4, ch_type='eeg', axes=axs[i])
    return all_importances, info_subj



In [ ]:
mne.set_log_level('ERROR')

In [ ]:
fig = compare_top_k_and_fractal(top_k_per_subject_abs, highest_corrs_fractal_offset_abs, highest_corrs_fractal_exponent_abs, subfig=None, sig_level=(0.05/60))

In [ ]:
fig.savefig("top_k_and_fractal_exponent_offset.png", dpi=300, bbox_inches='tight')

In [ ]:
compute_correlation_summary(all_subjects_highest_corrs_all_offset, significance_level=(0.05/60))

In [ ]:
fig.show()

## for frequency band gamma

In [ ]:
fig = plt.figure() 
subfigs = fig.subfigures(nrows=4, ncols=1)
all_importances, info = compare_top_k_and_significant2(top_10_per_subject_dict_abs, top_10_per_subject_dict, top_10_channels_abs_power, top_10_channels_relative_power, top_10_channels_fractal, top_10_channels_periodic,  frequency_band='gamma')

note for the gamma band absolute and relative power features channels in the sensorimotor region do not seem to be of importance often. However, fpr the fractal component (which is independent of frequency band) and periodic component channels in the sensorimoro region more often seem to be of importance.

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
mne.viz.plot_topomap(all_importances[5], info_subj , show = False, contours=4, ch_type='eeg', axes=ax)
fig.savefig(f"Average top 10 channels across subjects.png")

mabye get the channels with top-k correlation instead?

Also note that the premotoric areal is considered important across subjects? (FC3, AF3, AF4 is important often in the lower subjects)

Grouping subjects by most important channels may be interesting

investigate what happens when you remove the most important channels from the input


## for frequency band beta

In [ ]:
all_importances, info = compare_top_k_and_significant2(top_10_per_subject_dict_abs, top_10_per_subject_dict, top_10_channels_abs_power, top_10_channels_relative_power, top_10_channels_fractal, top_10_channels_periodic,  frequency_band='beta')

in the gamma bad the expected channels (C3) and nearby channels very often are part of the most important channels, suggesting that beta band statistics in this region are often often correlation with predicted amplitudes

## for frequency band alpha

In [ ]:
all_importances, info = compare_top_k_and_significant2(top_10_per_subject_dict_abs, top_10_per_subject_dict, top_10_channels_abs_power, top_10_channels_relative_power, top_10_channels_fractal, top_10_channels_periodic,  frequency_band='alpha')

similar to the beta band, features in the alpha band often are important in the sensorimotor region

## for frequency band theta

Interestingly even for the theta band the channels of interest see, to be among the most important channels often(i.e. aross subjects)

The gamma band seems to capture very different information compared to the other channels juding from this analysis. Is this why feeding the model information from only the beta and gamma band achieves such high accuracy?
To test this we would need to feed model information from only alpha and gamma band and check result on accuracy. Can this be done without introducing significant artifacts in the model?

Also check frequency ROAR results again when model was fed information from only alpha and beta band and only theta and alpha band.

In [ ]:
freq_bands = ["delta","theta", "alpha", "beta", "gamma"]
#fig = plt.figure(figsize=(12, 20))
fig = plt.figure(figsize=(12, 15))
fig.suptitle('Average topomaps of significant channels', fontsize=20,y=1.05, fontweight='bold')
subfigs = fig.subfigures(nrows=len(freq_bands), ncols=1)
for row, subfig in enumerate(subfigs):
    subfig.subplots_adjust(wspace=0.06)
    subfig.suptitle(f"frequency band {freq_bands[row]}", fontsize=18, y=0.98, fontweight='bold')
    all_importances, info = compare_power_features_all_channels(highest_corss_absolute_power_abs, highest_corss_relative_power_abs, highest_corrs_periodic_abs, frequency_band=freq_bands[row], subfig=subfig, sig_level=(0.05/60))
    
fig.savefig("all_power_features_average_new.png", bbox_inches='tight', dpi=300)


In [ ]:
fig.savefig("all_power_features_average.png", bbox_inches='tight', dpi=300)

In [ ]:
compare_top_k_and_fractal(top_k_per_subject_abs, highest_corrs_fractal_offset_abs, highest_corrs_fractal_exponent_abs, subfig=None, sig_level=(0.05/60))

In [ ]:
freq_bands = ["delta", "theta", "alpha", "beta", "gamma"]
#fig = plt.figure(figsize=(12, 20))
#fig = plt.figure()
#fig.suptitle('Most important channels', fontsize=16)

for band in freq_bands:
    all_importances, info = compare_power_features_all_channels(highest_corss_absolute_power_abs, highest_corss_relative_power_abs, highest_corrs_periodic_abs, frequency_band=band)
#fig.savefig("all_power_features_average.png")

In [ ]:
freq_bands = ["theta", "alpha", "beta", "gamma"]
#fig = plt.figure(figsize=(12, 20))
#fig = plt.figure()
#fig.suptitle('Most important channels', fontsize=16)

highest_corrs_absolute_power 
for band in freq_bands:
    all_importances, info = compare_power_features_all_channels(highest_corrs_absolute_power, highest_corrs_relative_power, highest_corrs_periodic, frequency_band=band)

In [ ]:
freq_bands = ["theta", "alpha", "beta", "gamma"]
#fig = plt.figure(figsize=(12, 20))
#fig = plt.figure()
#fig.suptitle('Most important channels', fontsize=16)

for band in freq_bands:
    all_importances, info = compare_power_features(top_10_channels_abs_power, top_10_channels_relative_power, top_10_channels_periodic, frequency_band=band)
#fig.savefig("all_power_features_average.png")

In [ ]:
#fig, ax = plt.subplots()
compare_top_k_and_fractal(top_10_per_subject_dict_abs, top_10_per_subject_dict, top_10_channels_fractal)


# compute rank correlations between significant channels and top k

get statistics of rank correlations between feature-amplitude correlations and and top-k channels according to the explanation function
Note that these correlation statistics are often low and only significant for very few subjects, so the explanation function may not be adequately explained by them

In the thesis write a segment about why we even want and explicit explanation function to work (fast approximation)

In [ ]:
#highest_corss_absolute_power = np.load('highest_corrs_absolute_power_abs.npy', allow_pickle=True).item()
#highest_corss_relative_power = np.load('highest_corrs_relative_power.npy', allow_pickle=True).item()
#highest_corrs_fractal = np.load('highest_corrs_fractal.npy', allow_pickle=True).item()
#highest_corrs_periodic = np.load('all_subjects_highest_corrs_perodic.npy', allow_pickle=True).item()
#highest_corrs_channel_wise_connectivity = np.load("highest_corss_channel_wise_functional_connectivity.npy", allow_pickle=True).item()
#top_k_per_subject = np.load("top_k_dict.npy", allow_pickle=True).item()


In [ ]:
feature_list = [highest_corrs_absolute_power, highest_corrs_relative_power, highest_corrs_periodic]
#highest_corrs_fractal
feature_names = ["absolute power", "relative power", "periodic component"]

In [ ]:
def compute_rank_correlations_features(cfg, freq_bands, highest_corrs, feature_name=""):
    """
    Compute rank correlations between subjects' stats for each frequency band.
    
    Parameters:
    -----------
    cfg : object
        Configuration object containing test_subject_indices
    freq_bands : dict
        Dictionary of frequency bands
    highest_corrs : dict
        Dictionary containing correlation data for each subject and frequency band
    feature_name : str
        Name of the feature being analyzed (for printing purposes)
        
    Returns:
    --------
    dict
        Dictionary containing average rank correlations for each frequency band
    """
    
    rank_correlations = {}
    for freq_band in freq_bands.keys():
        rank_correlations[freq_band] = {}
        
        # For each pair of subjects
        for i, subject1 in enumerate(cfg.dataset.test_subject_indices):
            for j, subject2 in enumerate(cfg.dataset.test_subject_indices):
                # Skip self-correlations and redundant pairs
                if subject1 >= subject2:
                    continue
                
                # Extract channel stats for both subjects
                stats1 = [highest_corrs[subject1][freq_band][ch]["stat"] 
                         for ch in highest_corrs[subject1][freq_band].keys()]
                stats2 = [highest_corrs[subject2][freq_band][ch]["stat"]
                         for ch in highest_corrs[subject2][freq_band].keys()]
                
                # Compute Spearman rank correlation
                corr, pval = spearmanr(stats1, stats2)
                rank_correlations[freq_band][(subject1, subject2)] = (corr, pval)
        
        # Calculate average correlation for this frequency band
        correlations = [x[0] for x in rank_correlations[freq_band].values()]
        avg_corr = np.mean(correlations)
        std_corr = np.std(correlations)
        
        print(f"{feature_name} - Frequency band {freq_band}:")
        print(f"Average rank correlation: {avg_corr:.3f} ± {std_corr:.3f}")
    
    return rank_correlations, avg_corr, std_corr


In [ ]:
_, avg_absolute, std_absolute = compute_rank_correlations_features(cfg, freq_bands, highest_corrs_absolute_power)

In [ ]:
_, avg_relative, std_relative = compute_rank_correlations_features(cfg, freq_bands, highest_corrs_relative_power)

In [ ]:
_, avg_periodic, std_periodic = compute_rank_correlations_features(cfg, freq_bands, highest_corrs_periodic)

In [ ]:
feature_list_abs = [highest_corss_absolute_power_abs,
highest_corss_relative_power_abs,
highest_corrs_periodic_abs]
#highest_corrs_fractal

In [ ]:
from scipy.stats import spearmanr
cfg = load_config()
def compute_rank_correlations(cfg, freq_bands, top_60_per_subject, corrs_per_freqband):
    rank_correlations = {}

    for freq_band in freq_bands.keys():
        rank_correlations[freq_band] = {}
        for subject_index in cfg.dataset.test_subject_indices:
            top_60_channels = top_60_per_subject[subject_index]
            #corrs = corrs_per_freqband[subject_index][freq_band]["stat"]
            corrs_per_freqband_values = [corrs_per_freqband[subject_index][freq_band][ch]["stat"] for ch in corrs_per_freqband[subject_index][freq_band].keys()]
            #print(subject_index)

            # Compute the Spearman rank correlation
            rank_corr, pval = spearmanr(list(top_60_channels.values()), corrs_per_freqband_values )
            #print(f"Rank correlation: {rank_corr:.2f}, p-value: {pval:.2f}")

            rank_correlations[freq_band][subject_index] = (rank_corr, pval)
    
    return rank_correlations
# Example usage:
# rank_correlations = compute_rank_correlations(cfg, freq_bands, top_60_per_subject, corrs_per_freqband)


In [ ]:
def compute_rank_correlations_features(cfg, freq_bands, corrs_per_freqband1, corrs_per_freqband2):
    rank_correlations = {}

    for freq_band in freq_bands.keys():
        rank_correlations[freq_band] = {}
        
        # Get list of subjects
        subjects = list(corrs_per_freqband1.keys())
        
        # Compute correlations for each pair of subjects
        for i, subject1 in enumerate(subjects):
            for j, subject2 in enumerate(subjects):
                # Skip correlations between same subject
                if subject1 >= subject2:
                    continue
                    
                # Extract values from first subject
                corrs1 = [corrs_per_freqband1[subject1][freq_band][ch]["stat"] 
                         for ch in corrs_per_freqband1[subject1][freq_band].keys()]
                
                # Extract values from second subject
                corrs2 = [corrs_per_freqband2[subject2][freq_band][ch]["stat"]
                         for ch in corrs_per_freqband2[subject2][freq_band].keys()]

                # Compute the Spearman rank correlation
                rank_corr, pval = spearmanr(corrs1, corrs2)

                # Store as tuple of (subject1, subject2)
                rank_correlations[freq_band][(subject1, subject2)] = (rank_corr, pval)

    # Compute average rank correlation across subject pairs for each frequency band
    avg_rank_correlations = {}
    std_rank_correlations = {}
    for freq_band in freq_bands.keys():
        correlations = [x[0] for x in rank_correlations[freq_band].values()]
        avg_rank_correlations[freq_band] = np.mean(correlations)
        std_rank_correlations[freq_band] = np.std(correlations)
        print(f"Frequency band {freq_band}:")
        print(f"Average rank correlation: {avg_rank_correlations[freq_band]:.3f} ± {std_rank_correlations[freq_band]:.3f}")

    return rank_correlations, avg_rank_correlations, std_rank_correlations


In [ ]:
freq_bands = {
    "delta" : (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma": (30, 45)
}

In [ ]:
top_60_per_subject_dict = top_k_per_subject
top_60_per_subject_abs_dict = top_k_per_subject_abs

In [ ]:
rank_correlations = {}
for feature, feature_name in zip(feature_list, feature_names):
    rank_correlation = compute_rank_correlations(cfg, freq_bands, top_60_per_subject_dict, feature)
    rank_correlations[feature_name] = rank_correlation


In [ ]:
rank_correlations_abs = {}
for feature, feature_name in zip(feature_list_abs, feature_names):
    rank_correlation = compute_rank_correlations(cfg, freq_bands, top_60_per_subject_abs_dict, feature)
    rank_correlations_abs[feature_name] = rank_correlation


In [ ]:
def plot_rank_correlations(rank_correlations, frequency_band='gamma', significance_level=0.05):
    fig, ax = plt.subplots(figsize=(15, 10))
    
    subjects = list(rank_correlations[list(rank_correlations.keys())[0]][frequency_band].keys())
    x = np.arange(len(subjects))
    width = 0.2  # the width of the bars
    
    for i, (feature_name, correlations) in enumerate(rank_correlations.items()):
        rank_corrs = [correlations[frequency_band][subject][0] for subject in subjects]
        pvals = [correlations[frequency_band][subject][1] for subject in subjects]
        
        ax.bar(x + i * width, rank_corrs, width, label=f'{feature_name}')
        
        for k, pval in enumerate(pvals):
            if pval < significance_level:
                ax.text(x[k] + i * width, rank_corrs[k], '*', ha='center', va='bottom', color='red', fontsize=12)
    
    ax.set_title(f'Rank Correlations in {frequency_band} band')
    ax.set_xlabel('Subject')
    ax.set_ylabel('Rank Correlation')
    ax.set_xticks(x + width * (len(rank_correlations) - 1) / 2)
    ax.set_xticklabels(subjects)
    ax.legend()
    
    plt.tight_layout()
    plt.show()

# Example usage:
plot_rank_correlations(rank_correlations, frequency_band='gamma')


In [ ]:
def plot_rank_correlations(rank_correlations, significance_level=0.05):
    frequency_bands = list(rank_correlations[list(rank_correlations.keys())[0]].keys())
    num_bands = len(frequency_bands)
    
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(15, 10), sharex=True, sharey=True)
    
    mean_correlations = {}
    
    for i, frequency_band in enumerate(frequency_bands):
        ax = axs[i // 2, i % 2]
        subjects = list(rank_correlations[list(rank_correlations.keys())[0]][frequency_band].keys())
        x = np.arange(len(subjects))
        width = 0.2  # the width of the bars
        
        mean_correlations[frequency_band] = {}
        
        for j, (feature_name, correlations) in enumerate(rank_correlations.items()):
            rank_corrs = [correlations[frequency_band][subject][0] for subject in subjects]
            pvals = [correlations[frequency_band][subject][1] for subject in subjects]
            
            mean_corr = np.mean(rank_corrs)
            mean_correlations[frequency_band][feature_name] = mean_corr
            
            ax.bar(x + j * width, rank_corrs, width, label=f'{feature_name} (mean={mean_corr:.2f})')
            
            for k, pval in enumerate(pvals):
                if pval < significance_level:
                    ax.text(x[k] + j * width, rank_corrs[k], '*', ha='center', va='bottom', color='red', fontsize=12)
        
        ax.set_title(f'rank correlations in {frequency_band} band')
        ax.set_xlabel('subject')
        ax.set_ylabel('rank correlation')
        ax.set_xticks(np.arange(len(cfg.dataset.test_subject_indices)))
        ax.set_xticklabels(np.arange(len(cfg.dataset.test_subject_indices)),rotation=45)
        ax.legend()
    fig.savefig("rank_correlations_top_k_abs_all_features.png")
    plt.tight_layout()
    plt.show()
    
    return mean_correlations

# Example usage:
#mean_correlations = plot_rank_correlations(rank_correlations)
#print(mean_correlations)

In [ ]:
def plot_rank_correlations(rank_correlations, significance_level=(0.05/3), label="abs"):
    frequency_bands = list(rank_correlations[list(rank_correlations.keys())[0]].keys())
    
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(15, 10), sharex=True, sharey=True)
    fig.subplots_adjust(wspace=0.05, hspace=0.15)  # Reduce space between subplots further
    
    colors = ['#009E73','#0072B2', '#D55E00'] 
    mean_correlations = {}
    std_correlations = {}
    
    for i, frequency_band in enumerate(frequency_bands):
        ax = axs[i // 2, i % 2]
        subjects = list(rank_correlations[list(rank_correlations.keys())[0]][frequency_band].keys())
        x = np.arange(len(subjects))
        width = 0.25  # Slightly wider bars
        
        mean_correlations[frequency_band] = {}
        std_correlations[frequency_band] = {}
        
        for j, (feature_name, correlations) in enumerate(rank_correlations.items()):
            rank_corrs = [correlations[frequency_band][subject][0] for subject in subjects]
            pvals = [correlations[frequency_band][subject][1] for subject in subjects]
            
            mean_corr = np.mean(rank_corrs)
            std_corr = np.std(rank_corrs)
            mean_correlations[frequency_band][feature_name] = mean_corr
            std_correlations[frequency_band][feature_name] = std_corr
            
            bar = ax.bar(x + j * width, rank_corrs, width, 
                   color=colors[j], linewidth=0.5)
            
            # Add significance markers with larger font size
            for k, pval in enumerate(pvals):
                if pval < significance_level:
                    if rank_corrs[k] > 0:
                        ax.text(x[k] + j * width, rank_corrs[k], '*', ha='center', va='bottom', 
                           color='black', fontsize=14, fontweight='bold')
                    else:
                        ax.text(x[k] + j * width, rank_corrs[k], '*', ha='center', va='top', 
                           color='black', fontsize=14, fontweight='bold')
           
        
        # Improve axis appearance
        ax.set_title(f'{frequency_band} band', fontsize=18, fontweight='bold')
        ax.set_xlabel('Subject index', fontsize=16)
        if i%2 == 0:
            ax.set_ylabel('Rank correlation', fontsize=16)
        ax.set_xticks(x + width)
        ax.set_xticklabels(np.arange(len(cfg.dataset.test_subject_indices)), rotation=45, fontsize=14)
        ax.tick_params(axis='y', labelsize=12)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)  # Add zero line
        
        # Add separate legend for each subplot
        #ax.legend(loc='lower center', fontsize=15, frameon=True, facecolor='white', edgecolor='black')
    
    fig.suptitle('Rank Correlations between Power Features and Model Explanations', 
                fontsize=20, y=0.98, fontweight='bold')
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Make room for the title
    fig.savefig(f"rank_correlations_top_k_{label}_all_features.png", dpi=300, bbox_inches='tight')
    
    return mean_correlations, std_correlations

In [ ]:
# Create a new dictionary with desired order
reordered_rank_correlations = {
    "absolute power": rank_correlations["absolute power"],
    "relative power": rank_correlations["relative power"],
    "periodic component": rank_correlations["periodic component"],
    
}

# Plot with the reordered dictionary
means, std = plot_rank_correlations(reordered_rank_correlations, label="")

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import os

def plot_rank_correlation_histograms(rank_correlations, label="", significance_level=(0.05/3)):
    """
    Plot histograms of rank correlations between features and model explanations.
    
    Parameters:
    -----------
    rank_correlations : dict
        Dictionary with features as keys, containing correlations for each frequency band and subject
    label : str
        Label to add to the saved figure filename
    significance_level : float
        Threshold for statistical significance (default: 0.05/3, Bonferroni correction)
    """
    colors = ['#0072B2','#009E73', '#D55E00'] 
    features = list(rank_correlations.keys())
    frequency_bands = list(rank_correlations[features[0]].keys())
    
    # Handle different possible dimensions of the plot
    if len(frequency_bands) == 1 and len(features) == 1:
        fig, ax = plt.subplots(figsize=(8, 6))
        axs = np.array([[ax]])
    elif len(frequency_bands) == 1:
        fig, axs_row = plt.subplots(ncols=len(features), figsize=(4*len(features), 6), sharex=True, sharey=True)
        axs = np.array([axs_row])
    elif len(features) == 1:
        fig, axs_col = plt.subplots(nrows=len(frequency_bands), figsize=(8, 3*len(frequency_bands)), sharex=True, sharey=True)
        axs = np.array([[ax] for ax in axs_col])
    else:
        fig, axs = plt.subplots(nrows=len(frequency_bands), ncols=len(features), 
                            figsize=(4*len(features), 3*len(frequency_bands)),
                            sharex=True, sharey=True)
        axs = np.array(axs)  # Ensure axs is always a numpy array for consistent indexing
    
    mean_correlations = {}
    std_correlations = {}
    
    for i, freq_band in enumerate(frequency_bands):
        mean_correlations[freq_band] = {}
        std_correlations[freq_band] = {}
        
        for j, feature in enumerate(features):
            # Safely extract correlation values
            try:
                correlations = [rank_correlations[feature][freq_band][subject][0] 
                           for subject in rank_correlations[feature][freq_band].keys()]
                
                # Calculate statistics
                pvals = [rank_correlations[feature][freq_band][subject][1] 
                        for subject in rank_correlations[feature][freq_band].keys()]
                num_significant = sum(p < significance_level for p in pvals)
                total_subjects = len(pvals)
                mean_corr = np.mean(correlations)
                std_corr = np.std(correlations)
                
                mean_correlations[freq_band][feature] = mean_corr
                std_correlations[freq_band][feature] = std_corr
                
                # Plot histogram
                ax = axs[i, j]
                sns.histplot(correlations, kde=True, ax=ax, bins=8,
                             color=colors[j % len(colors)], alpha=0.7, edgecolor='white', linewidth=0.8)
                
                # Add mean line
                ax.axvline(x=mean_corr, color='black', linestyle='-', linewidth=1.5)
                
                # Add stats box
                stats_text = f"Mean: {mean_corr:.2f}\nSignificant: {num_significant}/{total_subjects}"
                ax.text(0.7, 0.95, stats_text, transform=ax.transAxes,
                       fontsize=12, va='top',
                       bbox=dict(facecolor='white', alpha=0.85, boxstyle='round,pad=0.5', edgecolor='gray'))
                
                # Style the plot
                ax.set_xlim(-0.6, 0.6)
                ax.grid(axis='y', alpha=0.3, linestyle='--')
                ax.xaxis.grid(False)  # Explicitly turn off vertical grid lines
                ax.tick_params(axis='both', labelsize=15)  # Slightly smaller font size
                
                # Set titles and labels
                if i == 0:
                    ax.set_title(feature, fontsize=16, fontweight='bold')
                if j == 0:
                    ax.set_ylabel(f"{freq_band} band", fontsize=16, fontweight='bold')
                else:
                    ax.set_ylabel('')
                    
                if i == len(frequency_bands)-1:
                    ax.set_xlabel('Correlation coefficient', fontsize=16)
                else:
                    ax.set_xlabel('')
            except Exception as e:
                print(f"Error plotting {feature} in {freq_band} band: {e}")
                ax = axs[i, j]
                ax.text(0.5, 0.5, f"Error plotting data", ha='center', va='center',
                       transform=ax.transAxes, fontsize=12)
    
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.2, wspace=0.1)
    
    fig.suptitle('Distribution of Rank Correlations between Power Features and Model Explanations', 
                fontsize=16, fontweight='bold', y=1.04)
    
    # Save the figure
    filename = "rank_correlation_histograms.png"
    if label:
        filename = f"rank_correlation_histograms_{label}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    
    return fig


In [ ]:
fig = plot_rank_correlation_histograms(reordered_rank_correlations, label="")
fig.savefig("rank_correlation_histograms_power.png", dpi=300, bbox_inches='tight')

In [ ]:
means, std

In [ ]:
import seaborn as sns
# Create a new dictionary with desired order
reordered_rank_correlations_abs = {
    "periodic component": rank_correlations_abs["periodic component"],
    "absolute power": rank_correlations_abs["absolute power"],
    "relative power": rank_correlations_abs["relative power"]
}

# Plot with the reordered dictionary
fig = plot_rank_correlation_histograms(reordered_rank_correlations_abs, label="abs")


In [ ]:
#plot_rank_correlation_histograms(reordered_rank_correlations_abs, label="")

In [ ]:
def get_min_max_stats_per_freq_band(correlation_data, sig_level=0.05/60):
    """
    Get minimum and maximum statistical values for each frequency band across all subjects and channels.
    Also compute mean absolute stat values and proportion of significant p-values per frequency band.
    
    Parameters:
    -----------
    correlation_data : dict
        Dictionary containing correlation data for each subject, frequency band, and channel
    sig_level : float
        Significance level threshold for p-values, default is Bonferroni corrected
    
    Returns:
    --------
    tuple
        (min_max_dict, mean_abs_stat_per_band, significant_prop_per_band)
    """
    # Initialize result dictionary
    min_max_dict = {}
    
    # Get all unique frequency bands
    all_freq_bands = set()
    for subject in correlation_data:
        for freq_band in correlation_data[subject]:
            all_freq_bands.add(freq_band)
    
    # Initialize dictionaries for stats per frequency band
    mean_abs_stat_per_band = {}
    significant_prop_per_band = {}
    
    # Initialize min and max values for each frequency band
    for freq_band in all_freq_bands:
        min_max_dict[freq_band] = {"min_stat": float('inf'), "max_stat": float('-inf')}
        mean_abs_stat_per_band[freq_band] = {"sum": 0, "count": 0}
        significant_prop_per_band[freq_band] = {"sig_count": 0, "total_count": 0}
    
    # Process all subjects and channels
    for subject in correlation_data:
        for freq_band in correlation_data[subject]:
            for channel in correlation_data[subject][freq_band]:
                # Get the stat value
                stat_value = correlation_data[subject][freq_band][channel]["stat"]
                p_value = correlation_data[subject][freq_band][channel]["pval"]
                
                # Skip nan values
                if np.isnan(stat_value) or np.isnan(p_value):
                    continue
                
                # Update min and max
                if stat_value < min_max_dict[freq_band]["min_stat"]:
                    min_max_dict[freq_band]["min_stat"] = stat_value
                if stat_value > min_max_dict[freq_band]["max_stat"]:
                    min_max_dict[freq_band]["max_stat"] = stat_value
                
                # Update sums for mean calculation per frequency band
                mean_abs_stat_per_band[freq_band]["sum"] += abs(stat_value)
                mean_abs_stat_per_band[freq_band]["count"] += 1
                
                # Count significant p-values per frequency band
                significant_prop_per_band[freq_band]["total_count"] += 1
                if p_value < sig_level:
                    significant_prop_per_band[freq_band]["sig_count"] += 1
    
    # Calculate means and proportions for each frequency band
    for freq_band in all_freq_bands:
        count = mean_abs_stat_per_band[freq_band]["count"]
        if count > 0:
            mean_abs_stat_per_band[freq_band] = mean_abs_stat_per_band[freq_band]["sum"] / count
        else:
            mean_abs_stat_per_band[freq_band] = 0
        
        total = significant_prop_per_band[freq_band]["total_count"]
        if total > 0:
            significant_prop_per_band[freq_band] = significant_prop_per_band[freq_band]["sig_count"] / len(cfg.dataset.test_subject_indices)
        else:
            significant_prop_per_band[freq_band] = 0
    
    return min_max_dict, mean_abs_stat_per_band, significant_prop_per_band

In [ ]:
get_min_max_stats_per_freq_band(highest_corrs_absolute_power, sig_level=0.05/60)

In [ ]:
get_min_max_stats_per_freq_band(highest_corrs_relative_power, sig_level=0.05/60)

In [ ]:
get_min_max_stats_per_freq_band(highest_corrs_periodic, sig_level=0.05/60)

In [ ]:
means_abs, stds_abs 

In [ ]:
plot_rank_correlations(rank_correlations, label="")

In [ ]:
_,avg_absolute_power,_ = compute_rank_correlations_features(cfg, freq_bands, highest_corrs_absolute_power, highest_corrs_absolute_power)

# premotor area 

In [ ]:
# Define the premotor area channels
premotor_channels = ['FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6']

# Create a dictionary of all channels that correspond to the premotor area
premotor_area_dict = {ch: ch for ch in ch_names if ch in premotor_channels}

print(premotor_area_dict)
# Count how many subjects have at least one of the premotor channels in their top-k channels
count_subjects_with_premotor_channels = sum(any(ch in premotor_channels for ch in top_k_per_subject[subject]) for subject in top_k_per_subject)

# For each subject, return which of the top k channels are in the premotor area
premotor_channels_in_top_k = {subject: [ch for ch in top_k_per_subject[subject] if ch in premotor_channels] for subject in top_k_per_subject}

print(f"Number of subjects with at least one premotor channel in their top-k channels: {count_subjects_with_premotor_channels}")
print("Premotor channels in top-k for each subject:")
fig, axs = plt.subplots(nrows=4, ncols=4, figsize=(20, 20))
fig.suptitle("Premotor Area Channels Highlighted for Each Subject")


for i, subject_index in enumerate(cfg.dataset.test_subject_indices):
    file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/frequency_power_data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
    epochs = mne.read_epochs(file_path)
    info_subj = epochs.info
    ax = axs[i // 4, i % 4]
    data = np.zeros(len(ch_names))

    for ch in premotor_channels_in_top_k[subject_index]:
        data[ch_names.index(ch)] = 1
    mne.viz.plot_topomap(data, info_subj, show=False, contours=0, axes=ax, cmap='Reds')
    ax.set_title(f"Subject {subject_index}")
    
fig.savefig("premotor_channels_in_top_k.png")



In [ ]:
mean_correlations = plot_rank_correlations(rank_correlations_abs)

all subjects have at least on channel associated with the premotor are in their top channels.
Also all subjects for which this is the case to have the significant channels in their left hemissphre

In [ ]:
# Create a function to compute and plot the rank correlations between pairs of features
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

def compute_and_plot_feature_correlations(feature_list, feature_names, freq_bands, cfg, top_k_dict=None, method='spearman'):
    """
    Compute and plot correlations between pairs of features.
    
    Parameters:
    -----------
    feature_list : list
        List of feature dictionaries
    feature_names : list
        List of feature names
    freq_bands : dict
        Dictionary of frequency bands
    cfg : object
        Configuration object
    top_k_dict : dict, optional
        Dictionary of top k features
    method : str, optional
        Correlation method: 'spearman' or 'pearson' (default: 'spearman')
    
    Returns:
    --------
    dict
        Dictionary of correlations
    """
    # Dictionary to store all the correlations
    all_correlations = {}
    
    # Choose correlation function based on method
    if method.lower() == 'pearson':
        corr_func = pearsonr
        corr_type = 'Pearson'
    else:
        corr_func = spearmanr
        corr_type = 'Spearman'
    
    # Compute correlations between each pair of features
    for i, (feat1, name1) in enumerate(zip(feature_list, feature_names)):
        for j, (feat2, name2) in enumerate(zip(feature_list, feature_names)):
            if i < j:  # Only compute for unique pairs
                # Remove the word "power" from feature names
                display_name1 = name1.replace(" power", "")
                display_name2 = name2.replace(" power", "")
                pair_name = f"{display_name1} vs {display_name2}"
                all_correlations[pair_name] = {}
                
                for freq_band in freq_bands.keys():
                    all_correlations[pair_name][freq_band] = {}
                    
                    for subject_index in cfg.dataset.test_subject_indices:
                        # Get channel names and values for both features
                        channels1 = feat1[subject_index][freq_band]
                        channels2 = feat2[subject_index][freq_band]
                            
                        # Find common channels
                        common_channels = set(channels1.keys()).intersection(set(channels2.keys()))
                            
                        # Extract values for common channels
                        values1 = [channels1[ch]["stat"] for ch in common_channels]
                        values2 = [channels2[ch]["stat"] for ch in common_channels]

                        #corrs_per_freqband_values = [corrs_per_freqband[subject_index][freq_band][ch]["stat"] for ch in corrs_per_freqband[subject_index]#[freq_band].keys()
                                
                        # Compute correlation
                        corr, pval = corr_func(values1, values2)
                        all_correlations[pair_name][freq_band][subject_index] = (corr, pval)
    
    # Prepare data for plots
    pair_names = list(all_correlations.keys())
    freq_band_names = list(freq_bands.keys())
    
    # Prepare data structure for plotting
    pair_data = {}
    pair_data_abs = {}
    
    for pair_name in pair_names:
        pair_data[pair_name] = {
            'mean_corrs': [],
            'std_errs': [],
            'sig_counts': []
        }
        pair_data_abs[pair_name] = {
            'mean_corrs': [],
            'std_errs': [],
            'sig_counts': []
        }
        
        for freq_band in freq_band_names:
            correlations_band = all_correlations[pair_name][freq_band]
            
            # Extract correlations and p-values
            subjects = list(correlations_band.keys())
            corrs = [correlations_band[s][0] for s in subjects]
            pvals = [correlations_band[s][1] for s in subjects]
            
            # Copy for absolute values
            corrs_abs = [abs(c) for c in corrs]
            
            # Remove NaN values
            #valid_corrs = [c for c in corrs if not np.isnan(c)]
            #valid_pvals = [p for p, c in zip(pvals, corrs) if not np.isnan(c)]
            
            #valid_corrs_abs = [c for c in corrs_abs if not np.isnan(c)]
            #valid_pvals_abs = [p for p, c in zip(pvals, corrs_abs) if not np.isnan(c)]
            
            # Calculate statistics for regular values
            mean_corr = np.mean(corrs)
            std_err = np.std(corrs, ddof=1) / np.sqrt(len(corrs)) 
            #sig_count = sum(1 for p in valid_pvals if p < 0.05)
            
            pair_data[pair_name]['mean_corrs'].append(mean_corr)
            pair_data[pair_name]['std_errs'].append(std_err)
            #pair_data[pair_name]['sig_counts'].append(sig_count)
            
            # Calculate statistics for absolute values
            mean_corr_abs = np.mean(corrs_abs)
            std_err_abs = np.std(corrs_abs, ddof=1) / np.sqrt(len(corrs_abs))
            #sig_count_abs = sum(1 for p in corrs_abs if p < 0.05)
            
            pair_data_abs[pair_name]['mean_corrs'].append(mean_corr_abs)
            pair_data_abs[pair_name]['std_errs'].append(std_err_abs)
            #pair_data_abs[pair_name]['sig_counts'].append(sig_count_abs)
    
    # Create the plots side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    
    # Width of a bar 
    bar_width = 0.8 / len(pair_names)
    
    # Position of bars on x-axis
    index = np.arange(len(freq_band_names))
    
    # Define a new color palette that's different from previous plots
    colors = [
        "#4e79a7", "#f28e2c", "#e15759", "#76b7b2", "#59a14f",
        "#edc949", "#af7aa1", "#ff9da7", "#9c755f", "#bab0ab",
        "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e",
        "#a6761d", "#666666", "#e41a1c", "#377eb8", "#4daf4a"
    ]
    
    # Create plot for regular values (left subplot)
    for i, (pair_name, data) in enumerate(pair_data.items()):
        position = index - 0.4 + (i + 0.5) * bar_width
        bars = ax1.bar(position, data['mean_corrs'], width=bar_width, 
                     yerr=data['std_errs'], capsize=3, 
                     color=colors[i % len(colors)], label=pair_name, alpha=0.8)
    
    # Create plot for absolute values (right subplot)
    for i, (pair_name, data) in enumerate(pair_data_abs.items()):
        position = index - 0.4 + (i + 0.5) * bar_width
        bars = ax2.bar(position, data['mean_corrs'], width=bar_width, 
                     yerr=data['std_errs'], capsize=3, 
                     color=colors[i % len(colors)], label=pair_name, alpha=0.8)
    
    # Set labels, title and legend for first subplot (regular values)
    ax1.set_title(f"{corr_type} Correlations", fontsize=22)
    ax1.set_ylabel(f'{corr_type} Correlation', fontsize=20)
    ax1.set_xticks(index)
    ax1.set_xticklabels(freq_band_names, fontsize=20)
    ax1.set_yticklabels([f'{tick:.2f}' for tick in ax1.get_yticks()], fontsize=20)
    ax1.set_ylim(-0.5, 0.8)
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Set labels, title and legend for second subplot (absolute values)
    ax2.set_title(f"{corr_type} Correlations absolute values", fontsize=22)
    #ax2.set_ylabel(f'Absolute {corr_type} Correlation', fontsize=20)
    ax2.set_xticks(index)
    ax2.set_xticklabels(freq_band_names, fontsize=20)
    ax2.set_yticklabels([f'{tick:.2f}' for tick in ax2.get_yticks()], fontsize=20)
    #ax2.set_ylim(0, 1)
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Only need one legend for both plots
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.05), 
               ncol=len(pair_names), fontsize=22)
    
    plt.subplots_adjust(bottom=0.1)  # Make room for the legend at the bottom
    fig.savefig("feature_pair_correlations.png")
    plt.show()
    
    
    return all_correlations

# Call the function with the feature list
compute_and_plot_feature_correlations(feature_list, feature_names, freq_bands, cfg, method='spearman')

In [ ]:
# Create a function to compute and plot the rank correlations between pairs of features
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

def compute_and_plot_feature_correlations(feature_list, feature_names, freq_bands, cfg, top_k_dict=None, method='spearman', use_absolute=False):
    """
    Compute and plot correlations between pairs of features.
    
    Parameters:
    -----------
    feature_list : list
        List of feature dictionaries
    feature_names : list
        List of feature names
    freq_bands : dict
        Dictionary of frequency bands
    cfg : object
        Configuration object
    top_k_dict : dict, optional
        Dictionary of top k features
    method : str, optional
        Correlation method: 'spearman' or 'pearson' (default: 'spearman')
    use_absolute : bool, optional
        Whether to use absolute correlation values (default: False)
    
    Returns:
    --------
    dict
        Dictionary of correlations
    """
    # Dictionary to store all the correlations
    all_correlations = {}
    
    # Choose correlation function based on method
    if method.lower() == 'pearson':
        corr_func = pearsonr
        corr_type = 'Pearson'
    else:
        corr_func = spearmanr
        corr_type = 'Spearman'
    
    # Compute correlations between each pair of features
    for i, (feat1, name1) in enumerate(zip(feature_list, feature_names)):
        for j, (feat2, name2) in enumerate(zip(feature_list, feature_names)):
            if i < j:  # Only compute for unique pairs
                # Remove the word "power" from feature names
                display_name1 = name1.replace(" power", "")
                display_name2 = name2.replace(" power", "")
                pair_name = f"{display_name1} vs {display_name2}"
                all_correlations[pair_name] = {}
                
                for freq_band in freq_bands.keys():
                    all_correlations[pair_name][freq_band] = {}
                    
                    for subject_index in cfg.dataset.test_subject_indices:
                        # Get channel names and values for both features
                        channels1 = feat1[subject_index][freq_band]
                        channels2 = feat2[subject_index][freq_band]
                            
                        # Find common channels
                        common_channels = set(channels1.keys()).intersection(set(channels2.keys()))
                            
                        # Extract values for common channels
                        values1 = [channels1[ch]["stat"] for ch in common_channels]
                        values2 = [channels2[ch]["stat"] for ch in common_channels]
                                
                        # Compute correlation
                        corr, pval = corr_func(values1, values2)
                        all_correlations[pair_name][freq_band][subject_index] = (corr, pval)
    
    # Prepare data for plots
    pair_names = list(all_correlations.keys())
    freq_band_names = list(freq_bands.keys())
    
    # Prepare data structure for plotting
    pair_data = {}
    
    for pair_name in pair_names:
        pair_data[pair_name] = {
            'mean_corrs': [],
            'std_errs': [],
            'sig_counts': []
        }
        
        for freq_band in freq_band_names:
            correlations_band = all_correlations[pair_name][freq_band]
            
            # Extract correlations and p-values
            subjects = list(correlations_band.keys())
            corrs = [correlations_band[s][0] for s in subjects]
            pvals = [correlations_band[s][1] for s in subjects]
            
            # Apply absolute value if requested
            if use_absolute:
                corrs = [abs(c) for c in corrs]
            
            # Calculate statistics
            mean_corr = np.mean(corrs)
            std_err = np.std(corrs, ddof=1) / np.sqrt(len(corrs))
            
            pair_data[pair_name]['mean_corrs'].append(mean_corr)
            pair_data[pair_name]['std_errs'].append(std_err)
    
    # Create the plot
    fig, ax = plt.figure(figsize=(14, 6)), plt.gca()
    
    # Width of a bar 
    bar_width = 0.8 / len(pair_names)
    
    # Position of bars on x-axis
    index = np.arange(len(freq_band_names))
    
    # Define a color palette
    colors = [
        "#4e79a7", "#f28e2c", "#e15759", "#76b7b2", "#59a14f",
        "#edc949", "#af7aa1", "#ff9da7", "#9c755f", "#bab0ab",
        "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e",
        "#a6761d", "#666666", "#e41a1c", "#377eb8", "#4daf4a"
    ]
    
    # Create bars for each pair
    for i, (pair_name, data) in enumerate(pair_data.items()):
        print(data["mean_corrs"])
        position = index - 0.4 + (i + 0.5) * bar_width
        bars = ax.bar(position, data['mean_corrs'], width=bar_width, 
                     yerr=data['std_errs'], capsize=3, 
                     color=colors[i % len(colors)], label=pair_name, alpha=0.8)
    
    # Title and labels
    title_text = f"{corr_type} Correlations"
    if use_absolute:
        title_text += " (Absolute Values)"
    
    ax.set_title(title_text, fontsize=22)
    ax.set_ylabel(f'{corr_type} Correlation', fontsize=20)
    ax.set_xticks(index)
    ax.set_xticklabels(freq_band_names, fontsize=20)
    ax.set_yticklabels([f'{tick:.2f}' for tick in ax.get_yticks()], fontsize=18)

    
    # Set y-limits based on whether we're using absolute values
    if use_absolute:
        ax.set_ylim(0, 1)
    else:
        ax.set_ylim(-0.2, 1)
    
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add legend
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.15), 
              ncol=min(len(pair_names), 3), fontsize=16)
    #fig.legend(fontsize=15)
    
    plt.subplots_adjust(bottom=0.2)  # Make room for the legend at the bottom
    
    # Save the figure
    filename = "feature_pair_correlations"
    if use_absolute:
        filename += "_abs"
    fig.savefig(f"{filename}.png")
    
    plt.show()
    
    return all_correlations

# Call the function with the feature list
compute_and_plot_feature_correlations(feature_list, feature_names, freq_bands, cfg, method='spearman', use_absolute=False)


In [ ]:
compute_and_plot_feature_correlations(feature_list, feature_names, freq_bands, cfg, use_abs=True)